# 3.1 趋势类指标

> **这一节讲什么？**  
> 我们要学三个「跟着价格走势走」的指标：移动均线（SMA/EMA）和 MACD。
> 它们的核心思想都是：**把价格的噪声过滤掉，看清楚背后的方向**。

## 学习目标
- 理解移动均线（SMA / EMA）的构造原理
- 掌握 MACD 的计算与信号解读
- 用均线金叉死叉构造交易信号

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (13, 5)

data = yf.download('AAPL', start='2022-01-01', end='2024-01-01', progress=False)
close = data['Close'].squeeze()
print('数据加载完成 ✅')

In [ ]:
import matplotlib.pyplot as plt

# 1. 设置系统自带的中文字体（这里使用黑体 SimHei）
plt.rcParams['font.sans-serif'] = ['SimHei']  # 如果你想用微软雅黑，可以改成 ['Microsoft YaHei']

# 2. 解决更换字体后，负号（-）显示为方块的问题
plt.rcParams['axes.unicode_minus'] = False

## 1. SMA — 简单移动均线

### 为什么需要它？

你每天看股票，会发现价格像锯齿一样上上下下，今天涨了明天跌，很难看出到底是在上涨还是下跌。

**移动均线的思路很简单：把过去 N 天的价格做个平均，得到一条平滑的曲线。**

就像你评价一个员工，不会只看他某一天的表现，而是看「过去一个月的平均表现」。均线就是在做同样的事——平均掉每天的噪声，让趋势浮现出来。

### 公式

$$SMA_n(t) = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}$$

就是把最近 $n$ 天的收盘价加起来除以 $n$，就这么简单。

### 局限性
- **滞后**：只用到历史价格，所以均线永远「慢半拍」，价格已经转头了它还没反应
- **权重均等**：10天前的价格和昨天的价格，在 SMA 里权重一样——这合理吗？（下面的 EMA 来解决这个）

### 金叉 / 死叉

一个经典用法是同时画两条均线，短期（如20日）和长期（如60日）：
- **金叉（买入信号）**：短期均线从下穿越长期均线 → 说明近期价格开始强于长期趋势，趋势可能向上
- **死叉（卖出信号）**：短期均线从上穿越长期均线 → 近期价格开始弱于长期趋势，趋势可能向下

In [ ]:
sma20 = close.rolling(20).mean()
sma60 = close.rolling(60).mean()

# 金叉/死叉信号
cross_up = (sma20 > sma60) & (sma20.shift(1) <= sma60.shift(1))   # 金叉
cross_dn = (sma20 < sma60) & (sma20.shift(1) >= sma60.shift(1))   # 死叉

fig, ax = plt.subplots()
ax.plot(close.index, close.values, label='收盘价', linewidth=1, alpha=0.7)
ax.plot(sma20.index, sma20.values, label='SMA20（短期均线）', linewidth=1.5, color='orange')
ax.plot(sma60.index, sma60.values, label='SMA60（长期均线）', linewidth=1.5, color='red')
ax.scatter(close.index[cross_up], close.values[cross_up], marker='^', color='green', s=120, zorder=5, label='金叉（买入）')
ax.scatter(close.index[cross_dn], close.values[cross_dn], marker='v', color='red', s=120, zorder=5, label='死叉（卖出）')
ax.set_title('SMA20 / SMA60 金叉死叉策略', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. EMA — 指数移动均线

### 为什么在 SMA 之外还需要 EMA？

SMA 有一个直觉上的问题：**它把 30 天前的价格和昨天的价格一视同仁**。但常识告诉我们，昨天发生的事比一个月前的事更有参考价值。

**EMA 的改进：给最近的价格更高的权重。** 时间越近，权重越高；时间越远，权重呈指数衰减（exponential decay，这就是名字里「指数」的来源）。

效果：EMA 线比 SMA 线更贴近最新价格，对趋势变化反应更快。

### 公式

$$EMA_t = \alpha \cdot P_t + (1-\alpha) \cdot EMA_{t-1}, \quad \alpha = \frac{2}{n+1}$$

直觉理解：今天的 EMA = 今天价格的一点点 + 昨天 EMA 的大部分。
（$\alpha$ 越大，越重视今天的变化，EMA 越灵敏）

In [ ]:
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()

fig, ax = plt.subplots()
ax.plot(close.index, close.values, label='收盘价', linewidth=1, alpha=0.5)
ax.plot(sma20.index, sma20.values, label='SMA20（慢，滞后）', linewidth=1.5, linestyle='--')
ax.plot(ema12.index, ema12.values, label='EMA12（快，更贴近价格）', linewidth=1.5)
ax.set_title('SMA vs EMA 对比 — EMA 对价格变化反应更快', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. MACD — 指数平滑异同移动平均线

### 为什么需要 MACD？

单条均线只能告诉你「现在的趋势大方向」，但**不能告诉你趋势的强度在加速还是减弱**。

MACD 的思路很精妙：**用两条 EMA 之间的差距来衡量趋势的动力。**

类比：把 MACD 想象成一辆车的「油门踩了多深」——
- 短期 EMA（12日）追得快，代表近期市场力量
- 长期 EMA（26日）走得慢，代表长期背景力量
- **MACD = 短期 - 长期**：差值为正，说明近期比长期强，市场在加速向上；差值为负，说明近期弱，向下的力量在加大

### 三条线是什么

$$MACD = EMA_{12} - EMA_{26} \quad \text{（主线，衡量趋势差距）}$$
$$Signal = EMA_9(MACD) \quad \text{（信号线，主线的「均线」，更平滑）}$$
$$Histogram = MACD - Signal \quad \text{（柱状图，两线距离，衡量动力是否在增强）}$$

### 实战信号
- **MACD 线上穿 Signal 线**：买入信号（上涨动力增强）
- **MACD 线下穿 Signal 线**：卖出信号（下跌动力增强）
- **Histogram 柱子越来越长**：趋势动力加速；**越来越短**：动力在减弱，可能要转头

In [ ]:
macd_line = ema12 - ema26
signal_line = macd_line.ewm(span=9, adjust=False).mean()
histogram = macd_line - signal_line

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# 价格图
ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue')
ax1.set_title('AAPL 收盘价', fontsize=12)
ax1.set_ylabel('价格')
ax1.grid(alpha=0.3)

# MACD 图
ax2.plot(macd_line.index, macd_line.values, label='MACD（主线）', linewidth=1.5, color='blue')
ax2.plot(signal_line.index, signal_line.values, label='Signal（信号线）', linewidth=1.5, color='orange')
colors = ['green' if v >= 0 else 'red' for v in histogram.values]
ax2.bar(histogram.index, histogram.values, color=colors, alpha=0.4, label='Histogram（动力柱）')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('MACD 指标', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('📌 交易信号：MACD 线从下穿越 Signal 线 → 买入信号（近期上涨动力超过长期平均）')

## 🎯 练习

1. 将 SMA 参数从 (20, 60) 改为 (5, 20)，观察金叉/死叉频率变化。（短均线参数越小，信号越多，但假信号也越多）
2. MACD 的直方图（Histogram）在什么情况下达到峰值？这意味着什么？
3. 实现一个简单规则：MACD 线上穿 Signal 线买入，下穿卖出，计算年化收益。

---
**下一节** → `02_momentum_indicators.ipynb`